# 17. Stacked Meta-Model — F6: a second-level classifier over method predictions + the latent space (FOC-211)

Phase F6 of FOC-174: **stacked generalization**. Every base arm's TRAIN predictions are produced
OUT-OF-FOLD (grouped 5-fold over customers), so each TRAIN row carries a base-model probability
from a model that never saw it; a meta model is trained on those OOF probabilities ⊕ the raw
tabular features ⊕ the fused 1280-d latent space, and evaluated on TEST through each base model's
full-TRAIN-fit TEST predictions. The meta model therefore never scores a base model's own
training rows, and never sees test rows during any fit.

**Pre-registered expectation (from F3/F4/F5):** on the PRIMARY axis (cohort-random,
customer-grouped; 13 test positives, chance 0.0133) *every* arm so far covers chance and the
latent space itself carries ~no detectable signal at 5.3k transactions. A stacked meta model reads
that same representation — it adds capacity, not new information — so the PRIMARY null is the
expected outcome, not a failure. The deliverable is the leak-free machinery: the OOF stack,
evaluated through the identical `fraud_pipeline` protocol, ready for more data.

**Variants (all in `stack.py`, same harness as the arms):**

| variant | meta model | note |
|---|---|---|
| `meta-attn` | per-arm embeddings + multi-head attention pooling (PRIMARY) | attention weights logged as diagnostics |
| `meta-ftt` | per-feature tokens + 2-layer transformer encoder | |
| `meta-logit` | standardized logistic regression, C swept on the meta-val carve | |
| `meta-blend` | zero-fit nanmean of available arm probabilities | the reference every learned variant must beat |
| `meta-xgb` | shallow histogram gradient-boosted trees | |
| `meta-attn-sens` | `meta-attn` re-fit without `latent-nn-dist` / `face-features` / `demo-features` | chronological axis only — **sensitivity, not a purity certificate** |

**Leakage discipline (asserted, not assumed):** every join runs on row identity
`(customer, timestamp, row_index)`; a per-fold disjointness probe asserts fold i's OOF
probabilities never come from a fit that saw fold i's rows; the single stratified 25% meta-val
carve (seed 42) is used both for early stopping / C selection AND the frozen threshold; no
wall-clock fields anywhere.

In [1]:
import os

for _var in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_var] = '1'  # BEFORE numpy/torch load: threaded BLAS/OpenMP is 1-ulp nondeterministic

import platform
import sys

import numpy as np
import pandas as pd
import sklearn

print('python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)
print('platform', platform.platform())

from pathlib import Path

for name in [
    '../data/all_trxns.csv',
    '../data/dim_customer.csv',
    '../data/face_embeddings.npz',
    '../data/demo_embeddings.npz',
    '../artifacts/stack/manifest__random-grouped.json',
    '../artifacts/stack/manifest__grouped.json',
    '../artifacts/stack/manifest__chronological.json',
]:
    path = Path(name)
    print('%-48s %s' % (name, 'OK' if path.exists() else 'MISSING'))

python 3.11.9 | numpy 2.4.6 | pandas 2.3.3 | sklearn 1.7.2
platform Windows-10-10.0.26200-SP0
../data/all_trxns.csv                            OK
../data/dim_customer.csv                         OK
../data/face_embeddings.npz                      OK
../data/demo_embeddings.npz                      OK
../artifacts/stack/manifest__random-grouped.json OK
../artifacts/stack/manifest__grouped.json        OK
../artifacts/stack/manifest__chronological.json  OK


In [2]:
import arms_fusion
import stack
from fraud_pipeline import (
    ARMS,
    DEFAULT_RESULTS_PATH,
    axis_split,
    load_enriched,
    load_results,
    print_comparison_table,
)

STACK_DIR = Path('../artifacts/stack')
PRIMARY = 'random-grouped'
AXIS_ORDER = ('random-grouped', 'grouped', 'chronological')

# Canonical enriched frame + labels from the unified runner (shared by every arm).
enriched, y = load_enriched()
print(
    'fraud txns: %d of %d (%.2f%%) across %d unique customers'
    % (int(y.sum()), len(y), 100 * y.mean(), enriched['customer'].nunique())
)
missing = arms_fusion.check_dependencies()
print('dependency probe:', missing if missing else 'latent caches OK (no encoder loads)')

BASE_ARMS = [arm for arm in ARMS if not ARMS[arm]['placeholder']]
BASE_VARIANTS = ['meta-attn', 'meta-ftt', 'meta-logit', 'meta-blend', 'meta-xgb']
assert len(BASE_ARMS) == 19, 'expected the 19 registered base arms'
assert all(v in stack.META_MODELS for v in BASE_VARIANTS), 'meta variants must be registered'
assert stack.SENS_VARIANT in stack.META_MODELS, 'sensitivity variant must be registered'

manifests = {axis: stack._load_manifest(STACK_DIR, axis) for axis in AXIS_ORDER}
for axis in AXIS_ORDER:
    ok = [a for a, e in manifests[axis]['arms'].items() if e.get('status') == 'ok']
    ks = sorted({e['k_used'] for e in manifests[axis]['arms'].values() if e.get('status') == 'ok'})
    assert len(ok) == 19, '%s: %d/19 arms cached' % (axis, len(ok))
    train_idx, _ = axis_split(axis, enriched, y)
    print('%s: 19 arms ok | k=%s | train %d rows, %d positives' % (
        axis, ks, len(train_idx), int(y.loc[train_idx].sum())))
print('meta variants registered: %s (+%s on chronological)' % (', '.join(BASE_VARIANTS), stack.SENS_VARIANT))

C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-211-f6\src\funs.py:197: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  trxns_data["timestamp"] = pd.to_datetime(


fraud txns: 91 of 5302 (1.72%) across 100 unique customers


dependency probe: latent caches OK (no encoder loads)
random-grouped: 19 arms ok | k=[5] | train 4325 rows, 78 positives
grouped: 19 arms ok | k=[5] | train 4471 rows, 80 positives
chronological: 19 arms ok | k=[5] | train 4241 rows, 67 positives
meta variants registered: meta-attn, meta-ftt, meta-logit, meta-blend, meta-xgb (+meta-attn-sens on chronological)


## Methods — leak-free stacked generalization over the 19 arms

- **Out-of-fold base predictions.** For each base arm, TRAIN rows are split into k=5 folds grouped
  by customer — a deterministic greedy assignment over sorted customer ids balancing fraud count,
  then row count, then fold index. Fold i's OOF probabilities come from a model fitted on the
  other k−1 folds only, with a per-fold row-identity disjointness probe asserting the fit never
  saw the rows it predicts. A separate full-TRAIN fit produces the TEST predictions.
- **Meta matrix.** `[19 arm OOF probas | 19 missing-arm mask flags | 116 raw tabular features |
  1280-d fused latent]` — the column order keeps the FTT variant's scalar tokens contiguous.
- **Missing-arm tolerance.** An arm skipped at base time contributes a NaN probability + a mask
  flag (attention and FTT read the mask; blend/logit/xgb see the fill). The chronological
  sensitivity variant instead drops three synthetic-modality arms entirely — labeled sensitivity,
  not a purity certificate.
- **Single meta-val carve.** ONE stratified 25% carve of TRAIN (seed 42) serves both model
  selection (torch early stopping on val PR-AUC; logistic C sweep) and the frozen best-F1
  threshold — then a one-shot frozen-threshold test evaluation through the runner's own metric +
  bootstrap functions.
- **Variants.** `meta-attn` (PRIMARY) pools per-arm tokens with multi-head attention and logs the
  mean attention per arm as diagnostics; `meta-ftt` tokenizes every scalar feature; `meta-logit`
  sweeps C on the carve; `meta-blend` is a zero-fit nanmean reference; `meta-xgb` is a shallow
  histogram tree. The torch variants seed python/numpy/torch (and cudnn.deterministic) before
  every fit.

In [3]:
# --- OOF cache integrity: fold math, deterministic folds, row identity, AC2 probes
for axis in AXIS_ORDER:
    train_idx, test_idx = axis_split(axis, enriched, y)
    ident_tr, ident_te = stack._identity_arrays(enriched, train_idx), stack._identity_arrays(enriched, test_idx)
    y_tr_arr = np.asarray(y.loc[train_idx])
    sub = pd.DataFrame({'customer': enriched.loc[train_idx, 'customer'].to_numpy(), 'y': y_tr_arr})
    cust_rows = sub.groupby('customer').size().to_dict()
    cust_frauds = sub.groupby('customer')['y'].sum().astype(int).to_dict()
    fit_lo, fit_hi, fold_pos_min = 10 ** 9, 0, 10 ** 9
    for arm_name, entry in sorted(manifests[axis]['arms'].items()):
        assert entry.get('status') == 'ok', '%s/%s: %s' % (axis, arm_name, entry.get('status'))
        fr = entry['fold_report']
        k = int(entry['k_used'])
        assert len(fr) == k
        for f in fr:
            assert f['fit_rows'] + f['fold_rows'] == len(train_idx), '%s/%s fold size math' % (axis, arm_name)
            assert f['fit_positives'] + f['fold_positives'] == int(y_tr_arr.sum()), '%s/%s fold positive math' % (axis, arm_name)
            fit_lo, fit_hi = min(fit_lo, f['fit_rows']), max(fit_hi, f['fit_rows'])
            fold_pos_min = min(fold_pos_min, f['fold_positives'])
        recomputed = stack.assign_folds(cust_rows, cust_frauds, k)
        assert recomputed == {c: int(f) for c, f in entry['fold_assignments'].items()},             '%s/%s: fold assignment is not the deterministic greedy one' % (axis, arm_name)
        with np.load(STACK_DIR / entry['npz']) as data:
            for name, arr in zip(('customer', 'timestamp_ns', 'row_index'), ident_tr):
                assert np.array_equal(data[name], arr), '%s/%s: cached OOF identity mismatch' % (axis, arm_name)
            for name, arr in zip(('test_customer', 'test_timestamp_ns', 'test_row_index'), ident_te):
                assert np.array_equal(data[name], arr), '%s/%s: cached TEST identity mismatch' % (axis, arm_name)
            folds = data['fold']
            assert np.array_equal(np.bincount(folds, minlength=k), np.array([f['fold_rows'] for f in fr])),                 '%s/%s: fold label coverage != fold_report' % (axis, arm_name)
            oof, test_proba = data['oof_proba'], data['test_proba']
            # the four anomaly scorers emit raw scores (distances ~0.9-1.5,
            # gmm negative log-likelihood ~-60..-5), not calibrated probas —
            # the protocol only needs finiteness + rank consistency
            assert np.isfinite(oof).all() and np.isfinite(test_proba).all(),                 '%s/%s: cached scores not finite' % (axis, arm_name)
            # AC2 leakage probe, per fold, on the cached arrays themselves
            for i in range(k):
                stack._assert_identity_disjoint(
                    tuple(a[folds == i] for a in ident_tr),
                    tuple(a[folds != i] for a in ident_tr),
                    '%s/%s fold %d' % (axis, arm_name, i),
                )
    print('%s: 19/19 npz identity-aligned | folds deterministic + disjoint | oof/test finite | fit sizes %d-%d, min fold positives %d' % (
        axis, fit_lo, fit_hi, fold_pos_min))

random-grouped: 19/19 npz identity-aligned | folds deterministic + disjoint | oof/test finite | fit sizes 3057-3882, min fold positives 14


grouped: 19/19 npz identity-aligned | folds deterministic + disjoint | oof/test finite | fit sizes 3234-3976, min fold positives 14


chronological: 19/19 npz identity-aligned | folds deterministic + disjoint | oof/test finite | fit sizes 2526-3776, min fold positives 10


In [4]:
# --- meta variants x axes (same call path as the CLI: cache -> matrix -> fit)
meta_rows = []
attn_models = {}
for axis in AXIS_ORDER:
    train_idx, test_idx = axis_split(axis, enriched, y)
    cache, contributing, _man = stack.load_stack_cache(axis, STACK_DIR, enriched, train_idx, test_idx)
    matrix, layout = stack.build_meta_matrix(enriched, train_idx, test_idx, cache, contributing)
    variants = list(BASE_VARIANTS) + ([stack.SENS_VARIANT] if axis == 'chronological' else [])
    for variant in variants:
        drop_arms = stack.SENS_DROP_ARMS if variant == stack.SENS_VARIANT else ()
        if drop_arms:
            matrix_v, layout_v = stack.build_meta_matrix(
                enriched, train_idx, test_idx, cache, contributing, drop_arms)
        else:
            matrix_v, layout_v = matrix, layout
        stack._seed_everything()
        row, model = stack.run_meta_on_split(variant, matrix_v, layout_v, y, train_idx, test_idx)
        row['axis'] = axis
        meta_rows.append(row)
        if variant in ('meta-attn', stack.SENS_VARIANT):
            attn_models[(axis, variant)] = (model, layout_v, drop_arms)
        print(
            '%-14s %-16s PR-AUC %.4f (CI %.4f, %.4f) vs chance %.4f (delta %+.4f) -> %s'
            % (axis, variant, row['pr_auc'], row['pr_auc_ci_low'], row['pr_auc_ci_high'],
               row['chance_level'], row['pr_auc'] - row['chance_level'],
               'covers chance' if row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']
               else 'separates'))
for axis in AXIS_ORDER:
    print_comparison_table(
        [r for r in meta_rows if r['axis'] == axis],
        title='%s — stacked meta variants (frozen threshold)' % axis,
    )
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

random-grouped meta-attn        PR-AUC 0.0240 (CI 0.0117, 0.0606) vs chance 0.0133 (delta +0.0107) -> covers chance


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-211-f6\src\stack.py:718: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = torch.nn.TransformerEncoder(layer, num_layers=2)


random-grouped meta-ftt         PR-AUC 0.0205 (CI 0.0111, 0.0449) vs chance 0.0133 (delta +0.0072) -> covers chance


random-grouped meta-logit       PR-AUC 0.0217 (CI 0.0095, 0.0582) vs chance 0.0133 (delta +0.0084) -> covers chance


...

chronological  meta-ftt         PR-AUC 0.0521 (CI 0.0212, 0.1560) vs chance 0.0226 (delta +0.0295) -> covers chance


chronological  meta-logit       PR-AUC 0.0213 (CI 0.0131, 0.0360) vs chance 0.0226 (delta -0.0014) -> covers chance


chronological  meta-blend       PR-AUC 0.0952 (CI 0.0312, 0.2050) vs chance 0.0226 (delta +0.0725) -> separates


chronological  meta-xgb         PR-AUC 0.1015 (CI 0.0268, 0.2273) vs chance 0.0226 (delta +0.0789) -> separates


chronological  meta-attn-sens   PR-AUC 0.1064 (CI 0.0381, 0.2186) vs chance 0.0226 (delta +0.0838) -> separates

== random-grouped — stacked meta variants (frozen threshold) ==
          axis        arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high     f1  recall_at_precision  frozen_threshold  n_features
random-grouped  meta-attn     ok              13        977        0.0133  0.0240         0.0117          0.0606   0.5789          0.3914           0.7637 0.0000                  0.0            0.9841        1434
random-grouped   meta-ftt     ok              13        977        0.0133  0.0205         0.0111          0.0449   0.6444          0.5383           0.7461 0.0000                  0.0            0.9972        1434
random-grouped meta-logit     ok              13        977        0.0133  0.0217         0.0095          0.0582   0.5496          0.3488           0.7224 0.0000                  0.0  

## Attention diagnostics — what the PRIMARY variant reads

`meta-attn` pools the 19 per-arm tokens with multi-head attention; the mean attention weight per
arm over the meta-val carve is the model's own account of which inputs it used. With 13 test
positives these weights are descriptive, not inferential — but they must be on the record,
especially for the sensitivity variant, whose whole point is how the pool re-balances when the
three synthetic-modality arms are removed.

In [5]:
import json

for (axis, variant), (model, layout_v, drop_arms) in sorted(attn_models.items()):
    weights = {arm: float(w) for arm, w in zip(layout_v.arm_names, model.attention_)}
    ranked = sorted(weights.items(), key=lambda kv: -kv[1])
    print('%s | %s | val PR-AUC %.4f, %d epochs%s' % (
        axis, variant, model.train_['val_pr_auc'], model.train_['epochs'],
        ' | drops: ' + ', '.join(drop_arms) if drop_arms else ''))
    for arm, w in ranked:
        print('  %-24s %.4f %s' % (arm, w, '#' * int(round(w * 100))))
    path = STACK_DIR / ('attn__%s__%s.json' % (axis, variant))
    if path.exists():
        payload = json.loads(path.read_text(encoding='utf-8'))
        assert payload['mean_attention'] == weights,             'attention diagnostics diverge from the CLI run (%s/%s)' % (axis, variant)
        print('  matches CLI diagnostics file attn__%s__%s.json (bitwise)' % (axis, variant))
    else:
        stack._save_attn_diagnostics(STACK_DIR, axis, variant, model, layout_v, drop_arms)
        print('  saved (first writer) attn__%s__%s.json' % (axis, variant))
    print()

chronological | meta-attn | val PR-AUC 0.5883, 38 epochs
  latent-gmm-density       0.4125 #########################################
  xgb-baseline             0.0505 #####
  dictionary               0.0488 #####
  latent-centroid-dist     0.0422 ####
  latent-nn-dist           0.0383 ####
  latent-cluster-anom      0.0380 ####
  face-features            0.0364 ####
  timesfm-features         0.0332 ###
  text-features-minilm-l6  0.0323 ###
  xgb-client               0.0310 ###
  latent-fusion            0.0306 ###
  demo-features            0.0301 ###
  tabnet                   0.0293 ###
  sequential-transformer   0.0273 ###
  latent-logistic          0.0272 ###
  sequential-lstm          0.0257 ###
  latent-cosine-centroid   0.0247 ##
  gbdt-ensemble            0.0224 ##
  sce                      0.0196 ##
  matches CLI diagnostics file attn__chronological__meta-attn.json (bitwise)

chronological | meta-attn-sens | val PR-AUC 0.5783, 31 epochs | drops: latent-nn-dist, face-features

## Comparison against the anchors + the best single arm

The meta model's whole question is whether a second level beats its own inputs. The anchors are
the conceptual ancestors from earlier phases (`dictionary`, `xgb-baseline`, `latent-fusion`); the
extra row is the strongest single arm on each axis from the accumulated runner results — the thing
stacking must actually beat for the machinery to matter.

In [6]:
prior = load_results(DEFAULT_RESULTS_PATH)
ANCHORS = ('dictionary', 'xgb-baseline', 'latent-fusion')
for axis in AXIS_ORDER:
    base_ok = [
        r for r in prior
        if r.get('axis') == axis and r.get('status') == 'ok'
        and not str(r.get('arm', '')).startswith('meta-')
    ]
    best_base = max(base_ok, key=lambda r: r['pr_auc']) if base_ok else None
    anchor_rows = [r for r in prior if r.get('axis') == axis and r.get('arm') in ANCHORS]
    combined = sorted(
        [r for r in meta_rows if r['axis'] == axis] + anchor_rows + ([best_base] if best_base else []),
        key=lambda r: (r['arm'] in ANCHORS, r['arm']),
    )
    if best_base is not None:
        print('strongest single arm on %s: %s (PR-AUC %.4f)' % (axis, best_base['arm'], best_base['pr_auc']))
    print_comparison_table(
        combined,
        title='%s — stacked meta vs anchors + best base arm (frozen threshold)' % axis,
    )
    print()

strongest single arm on random-grouped: dictionary (PR-AUC 0.0231)

== random-grouped — stacked meta vs anchors + best base arm (frozen threshold) ==
          axis           arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high     f1  recall_at_precision  frozen_threshold  cv_pr_auc_mean  cv_pr_auc_std  n_features
random-grouped  xgb-baseline     ok              13        977        0.0133  0.0171         0.0093          0.0350   0.5622          0.4083           0.7099 0.0000                  0.0            0.7978          0.5316         0.1130          87
random-grouped    dictionary     ok              13        977        0.0133  0.0231         0.0122          0.0469   0.6642          0.5209           0.7892 0.0000                  0.0            0.8355          0.5071         0.1027          94
random-grouped    dictionary     ok              13        977        0.0133  0.0231         0.0122          

In [7]:
# --- determinism: matrix rebuild + meta re-run + notebook/CLI bitwise contract
train_idx, test_idx = axis_split(PRIMARY, enriched, y)
cache, contributing, _man = stack.load_stack_cache(PRIMARY, STACK_DIR, enriched, train_idx, test_idx)
m1, l1 = stack.build_meta_matrix(enriched, train_idx, test_idx, cache, contributing)
m2, l2 = stack.build_meta_matrix(enriched, train_idx, test_idx, cache, contributing)
assert m1.index.equals(m2.index) and list(m1.columns) == list(m2.columns), 'indices differ - cannot align'
delta = float(np.abs(m1.to_numpy(dtype=np.float64) - m2.to_numpy(dtype=np.float64)).max())
print('identity-aligned meta matrix rebuild: max |delta| = %.1e over %d rows x %d cols' % (delta, m1.shape[0], m1.shape[1]))
assert delta == 0.0

for variant in ('meta-attn', 'meta-logit', 'meta-blend'):
    stack._seed_everything()
    r1, _model = stack.run_meta_on_split(variant, m1, l1, y, train_idx, test_idx)
    stack._seed_everything()
    r2, _model = stack.run_meta_on_split(variant, m1, l1, y, train_idx, test_idx)
    del r1['axis'], r2['axis']
    assert r1 == r2, '%s re-run diverged on %s' % (variant, PRIMARY)
    print('%s re-run: %d metric fields byte-identical (fixed seeds, no wall-clock fields)' % (variant, len(r1)))

persisted = {(r.get('axis'), r.get('arm')): r for r in load_results(DEFAULT_RESULTS_PATH)}
for row in meta_rows:
    p = persisted.get((row['axis'], row['arm']))
    assert p is not None, 'meta row %s/%s missing from the JSONL - run the CLI phase first' % (row['axis'], row['arm'])
    assert p == row, 'meta row %s/%s diverges from the persisted CLI row' % (row['axis'], row['arm'])
print('all %d meta rows byte-identical to results/fraud_pipeline_results.jsonl (CLI == notebook)' % len(meta_rows))

identity-aligned meta matrix rebuild: max |delta| = 0.0e+00 over 5302 rows x 1434 cols


meta-attn re-run: 19 metric fields byte-identical (fixed seeds, no wall-clock fields)


meta-logit re-run: 19 metric fields byte-identical (fixed seeds, no wall-clock fields)


meta-blend re-run: 19 metric fields byte-identical (fixed seeds, no wall-clock fields)
all 16 meta rows byte-identical to results/fraud_pipeline_results.jsonl (CLI == notebook)


## Honest interpretation

- **The pre-registered null is the expected result.** Stacking adds capacity, not information: the
  OOF arm probabilities on PRIMARY cover chance (F0-F4), and the latent block is the same
  representation F5 showed carries ~no signal at 5.3k transactions. A second-level model cannot
  manufacture signal its inputs do not contain; a PRIMARY separation would be an artifact to
  investigate first, a null is the honest reading.
- **What the machinery buys.** The leak-free OOF stack now exists inside the one runner protocol —
  new arms drop in as proba columns, skipped arms become mask flags, and every join is asserted on
  row identity with per-fold disjointness probes. More data can be dropped in and the whole layer
  re-compared without a parallel evaluation path.
- **Multiplicity, on the record.** 5 variants × 3 axes + 1 sensitivity fit = 16 reported fits over
  13 test positives; the best table row is a selection effect, not a discovery. The chronological
  sensitivity arm re-balances the attention pool when the synthetic-modality arms leave — it is
  labeled sensitivity, not a purity certificate.
- **Synthetic-data caveat.** Faces, text and demographics are simulated (F4). Conclusions are about
  method viability, not real-world fraud rates.
- **Scope.** F6 only; follow-ups are listed in the FOC-211 hand-off, not built here.